# Stickman Motion Studio — AI Script-Writer Benchmark
Compare local Qwen and Gemma models on one identical script-to-scenes task.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece pandas

In [ ]:
import os, json, time, gc, torch, pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
OUT=Path('/kaggle/working/stickman_script_benchmark'); OUT.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if torch.cuda.is_available(): print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

## Shared task
Both models receive exactly the same prompt.

In [ ]:
BRIEF='''You are the scriptwriter and visual director for Stickman Thoughts. Topic: What would happen if the Sun suddenly disappeared? Create a 30–60 second vertical 9:16 YouTube Short. Write an engaging hook and natural narration (90–125 words), logical escalation and payoff. Create 10–14 scenes. Mascot: large round white head with thick dark outline, two small black oval eyes, expressive eyebrows, small mouth, thin black stick arms and legs, simple body, no gloves or shoes. Flat 2D cartoon, clean thick outlines, soft muted colours. Keep identity, costume, props and locations consistent. Return JSON only with title, hook, estimated_duration_seconds, narration, claims_to_verify (claim and why_check), cast_bible, continuity_bible, scenes and final_continuity_checklist. Every scene must include scene_number, duration_seconds, narration_segment, action, expression, camera, composition, environment, lighting, continuity_from_previous and a detailed independently usable image_prompt. Do not put text, captions or watermarks inside images. Identify facts needing verification; do not claim they are verified.'''
CANDIDATES={'Qwen3-4B-Instruct-2507':'Qwen/Qwen3-4B-Instruct-2507','Gemma-3-4B-IT':'google/gemma-3-4b-it'}
MAX_NEW_TOKENS=5000

## Run candidates sequentially
Select a GPU accelerator and enable Internet. Accept Gemma access terms if prompted. If a model fails to load, the error is recorded rather than silently substituting another model.

In [ ]:
def run_model(label, model_id):
    started=time.time(); r={'model':label,'model_id':model_id}
    try:
        tok=AutoTokenizer.from_pretrained(model_id)
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype,device_map='auto',low_cpu_mem_usage=True)
        msgs=[{'role':'user','content':BRIEF}]
        prompt=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True) if tok.chat_template else 'User: '+BRIEF+'\nAssistant:'
        inputs=tok(prompt,return_tensors='pt').to(model.device)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        with torch.inference_mode(): out=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.05)
        raw=tok.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)
        r['seconds']=round(time.time()-started,2); r['peak_gpu_gib']=round(torch.cuda.max_memory_allocated()/1024**3,2) if torch.cuda.is_available() else None
        (OUT/(label+'.txt')).write_text(raw,encoding='utf-8'); r['raw_output']=raw
        try:
            clean=raw.strip()
            if clean.startswith('```'): clean='\n'.join(clean.splitlines()[1:-1])
            data=json.loads(clean); scenes=data.get('scenes',[])
            required=['scene_number','duration_seconds','narration_segment','action','expression','camera','composition','environment','lighting','continuity_from_previous','image_prompt']
            checks={'json_valid':True,'hook':bool(data.get('hook')),'narration':bool(data.get('narration')),'scene_count_10_14':10<=len(scenes)<=14,'scene_fields_complete':bool(scenes) and all(all(str(s.get(k,'')).strip() for k in required) for s in scenes),'claims_to_verify':isinstance(data.get('claims_to_verify'),list),'cast_bible':bool(data.get('cast_bible')),'continuity_bible':isinstance(data.get('continuity_bible'),dict),'final_checklist':isinstance(data.get('final_continuity_checklist'),list)}
            r['checks']=checks; r['score']=round(100*sum(checks.values())/len(checks)); (OUT/(label+'.json')).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        except Exception as e: r['score']=0; r['checks']={'json_valid':False}; r['parse_error']=str(e)
    except Exception as e: r['error']=repr(e); r['seconds']=round(time.time()-started,2)
    finally:
        for v in ['model','tok','inputs','out']:
            if v in locals(): del locals()[v]
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return r

results=[]
for label,mid in CANDIDATES.items():
    item=run_model(label,mid); results.append(item)
    (OUT/'benchmark_results.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')
    print(label,'score=',item.get('score'),'time=',item.get('seconds'),'error=',item.get('error'))

## Comparison report
Structural score checks completeness only. Manually judge writing quality, factual caution, scene variety, prompt specificity and character continuity.

In [ ]:
rows=[{'model':r['model'],'model_id':r['model_id'],'score':r.get('score'),'seconds':r.get('seconds'),'peak_gpu_gib':r.get('peak_gpu_gib'),'error':r.get('error',r.get('parse_error',''))} for r in results]
df=pd.DataFrame(rows); display(df); df.to_csv(OUT/'model_comparison.csv',index=False)
print('Saved:',OUT)
print('Important: model-generated claims are not fact-checked. Verify sources before publishing.')